In [ ]:
!pip install tensorflow keras numpy matplotlib pillow nltk tqdm

In [ ]:
import os, zipfile, urllib.request
# Download Flickr8k images
urllib.request.urlretrieve(
    "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip",
    "Flickr8k_Dataset.zip"
)
urllib.request.urlretrieve(
    "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip",
    "Flickr8k_text.zip"
)
# Unzip both files
with zipfile.ZipFile("Flickr8k_Dataset.zip", "r") as z:
    z.extractall("flickr8k")
with zipfile.ZipFile("Flickr8k_text.zip", "r") as z:
    z.extractall("flickr8k")

print("Dataset ready!")

Dataset ready!


In [ ]:
import string, re
def load_captions(filepath):
    with open(filepath) as f:
        text = f.read()
    captions = {}
    for line in text.strip().split("\n"):
        parts = line.split("\t")
        if len(parts) < 2:
            continue
        img_id, caption = parts[0], parts[1]
        img_name = img_id.split("#")[0]
        # Clean caption
        caption = caption.lower()
        caption = caption.translate(str.maketrans("","",string.punctuation))
        caption = re.sub(r"\s+", " ", caption).strip()
        caption = "startseq " + caption + " endseq"
        captions.setdefault(img_name, []).append(caption)
    return captions

captions = load_captions("flickr8k/Flickr8k.token.txt")
print(f"Loaded captions for {len(captions)} images")
print("Example:", list(captions.values())[0][0])

Loaded captions for 8092 images
Example: startseq a child in a pink dress is climbing up a set of stairs in an entry way endseq


In [ ]:
import numpy as np
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tqdm import tqdm
import pickle
# Load ResNet50 without the final classification layer
base_model = ResNet50(weights="imagenet")
model = Model(inputs=base_model.input,
              outputs=base_model.layers[-2].output)
IMAGE_DIR = "flickr8k/Flicker8k_Dataset/"
def extract_features(image_dir):
    features = {}
    img_files = os.listdir(image_dir)
    for img_name in tqdm(img_files, desc="Extracting features"):
        img_path = os.path.join(image_dir, img_name)
        try:
            img = load_img(img_path, target_size=(224, 224))
            arr = img_to_array(img)
            arr = np.expand_dims(arr, axis=0)
            arr = preprocess_input(arr)
            feat = model.predict(arr, verbose=0)
            features[img_name] = feat
        except:
            pass
    return features
features = extract_features(IMAGE_DIR)
# Save features so we don't recompute every time
with open("features.pkl", "wb") as f:
    pickle.dump(features, f)
print(f"Extracted features for {len(features)} images")

102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Extracting features: 100%|██████████| 8091/8091 [13:10<00:00, 10.24it/s]


Extracted features for 8091 images


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

all_captions = [cap for caps in captions.values() for cap in caps]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(c.split()) for c in all_captions)

print(f"Vocabulary size: {vocab_size}")
print(f"Max caption length: {max_length} words")

import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

Vocabulary size: 8831
Max caption length: 38 words


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, Add

def build_model(vocab_size, max_length):
    img_input = Input(shape=(2048,))
    img_drop = Dropout(0.4)(img_input)
    img_dense = Dense(256, activation="relu")(img_drop)

    cap_input = Input(shape=(max_length,))
    # ← mask_zero removed (was causing the cuDNN mask error)
    cap_embed = Embedding(vocab_size, 256)(cap_input)
    cap_drop = Dropout(0.4)(cap_embed)
    # ← use_cudnn=False fixes the cuDNN right-padding assertion
    cap_lstm = LSTM(256, use_cudnn=False)(cap_drop)

    merged = Add()([img_dense, cap_lstm])
    merged = Dense(256, activation="relu")(merged)
    output = Dense(vocab_size, activation="softmax")(merged)

    model = Model(inputs=[img_input, cap_input], outputs=output)
    model.compile(loss="categorical_crossentropy", optimizer="adam")
    return model

caption_model = build_model(vocab_size, max_length)
caption_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 38)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 38, 256)   │  2,260,736 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2048)      │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 38, 256)   │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 256)       │    525,312 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 256)       │          0 │ dense[0][0],      │
│                     │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     65,792 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8831)      │  2,269,567 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,645,951 (21.54 MB)

 Trainable params: 5,645,951 (21.54 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
import tensorflow as tf

def make_dataset(captions_dict, features, tokenizer,
                 max_length, vocab_size, batch_size=64):

    img_ids = [k for k in captions_dict if k in features]

    def generator():
        for img_name in img_ids:
            feat = features[img_name][0]
            for caption in captions_dict[img_name]:
                seq = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(seq)):
                    # ← padding='post' = right-padding, required by cuDNN
                    in_seq = pad_sequences(
                        [seq[:i]], maxlen=max_length, padding='post'
                    )[0]
                    out_word = tf.keras.utils.to_categorical(
                        [seq[i]], num_classes=vocab_size
                    )[0]
                    yield (feat, in_seq), out_word

    output_signature = (
        (
            tf.TensorSpec(shape=(2048,),       dtype=tf.float32),
            tf.TensorSpec(shape=(max_length,), dtype=tf.int32),
        ),
        tf.TensorSpec(shape=(vocab_size,), dtype=tf.float32),
    )

    dataset = tf.data.Dataset.from_generator(
        generator, output_signature=output_signature
    )
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import ModelCheckpoint

# ── Fixed data generator using tf.data ──────────────────────────────────────

def make_dataset(captions_dict, features, tokenizer,
                 max_length, vocab_size, batch_size=64):

    img_ids = [k for k in captions_dict if k in features]

    def generator():
        for img_name in img_ids:
            feat = features[img_name][0]          # shape (2048,)
            for caption in captions_dict[img_name]:
                seq = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(seq)):
                    in_seq = tf.keras.preprocessing.sequence.pad_sequences(
                        [seq[:i]], maxlen=max_length
                    )[0]
                    out_word = tf.keras.utils.to_categorical(
                        [seq[i]], num_classes=vocab_size
                    )[0]
                    yield (feat, in_seq), out_word

    # Tell TensorFlow the exact shapes and types — this is what was missing
    output_signature = (
        (
            tf.TensorSpec(shape=(2048,),      dtype=tf.float32),   # image feat
            tf.TensorSpec(shape=(max_length,), dtype=tf.int32),    # partial caption
        ),
        tf.TensorSpec(shape=(vocab_size,),    dtype=tf.float32),   # next word
    )

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=output_signature
    )
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


# ── Build dataset and train ──────────────────────────────────────────────────

train_dataset = make_dataset(
    captions, features, tokenizer,
    max_length, vocab_size, batch_size=64
)

checkpoint = ModelCheckpoint(
    "best_model.h5",
    monitor="loss",
    save_best_only=True,
    verbose=1
)

caption_model.fit(
    train_dataset,
    epochs=20,
    callbacks=[checkpoint],
    verbose=1
)

print("Training complete! Model saved as best_model.h5")

Epoch 1/20
   7453/Unknown 275s 36ms/step - loss: 4.3056
Epoch 1: loss improved from None to 3.87259, saving model to best_model.h5


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: finished saving model to best_model.h5
7453/7453 ━━━━━━━━━━━━━━━━━━━━ 275s 36ms/step - loss: 3.8726
Epoch 2/20
7452/7453 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 3.2436
Epoch 2: loss improved from 3.87259 to 3.19009, saving model to best_model.h5



Epoch 2: finished saving model to best_model.h5
7453/7453 ━━━━━━━━━━━━━━━━━━━━ 267s 36ms/step - loss: 3.1901
Epoch 3/20
7453/7453 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 2.9938
Epoch 3: loss improved from 3.19009 to 2.97404, saving model to best_model.h5



Epoch 3: finished saving model to best_model.h5
7453/7453 ━━━━━━━━━━━━━━━━━━━━ 273s 37ms/step - loss: 2.9740
Epoch 4/20
7452/7453 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 2.8546
Epoch 4: loss improved from 2.97404 to 2.84959, saving model to best_model.h5



Epoch 4: finished saving model to best_model.h5
7453/7453 ━━━━━━━━━━━━━━━━━━━━ 274s 37ms/step - loss: 2.8496
Epoch 5/20
7451/7453 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 2.7615
Epoch 5: loss improved from 2.84959 to 2.76266, saving model to best_model.h5



Epoch 5: finished saving model to best_model.h5
7453/7453 ━━━━━━━━━━━━━━━━━━━━ 270s 36ms/step - loss: 2.7627
Epoch 6/20
1040/7453 ━━━━━━━━━━━━━━━━━━━━ 3:48 36ms/step - loss: 2.7078

In [ ]:
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
from PIL import Image

caption_model = load_model("best_model.h5")

def generate_caption(img_path, model, tokenizer,
                     max_length, feat_model):
    img = load_img(img_path, target_size=(224, 224))
    arr = img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr)
    feat = feat_model.predict(arr, verbose=0)

    caption = "startseq"
    for _ in range(max_length):
        seq = tokenizer.texts_to_sequences([caption])[0]
        seq = pad_sequences([seq], maxlen=max_length)
        pred = model.predict([feat, seq], verbose=0)
        word_idx = np.argmax(pred)
        word = tokenizer.index_word.get(word_idx, None)
        if word is None or word == "endseq":
            break
        caption += " " + word
    return caption.replace("startseq", "").strip()

# Test it!
test_img = "flickr8k/Flicker8k_Dataset/1000268201_693b08cb0e.jpg"
caption = generate_caption(test_img, caption_model, tokenizer,
                           max_length, model)
plt.imshow(Image.open(test_img))
plt.axis("off")
plt.title(caption, fontsize=12)
plt.show()